# OFBiz HR Jira: preprocessing before CSV export

This notebook follows the Lab 5 workflow: inspect the collected data, clean it, validate it, and **only then** save a final UTF-8 CSV. It does not scrape the website; first run `scrape_ofbiz_hr.py` to collect the raw Jira export.

The raw input is kept unchanged. The final file uses exactly the same columns and order as `dataset.csv`.

## 1. Setup

Install pandas once if it is not already available:

```powershell
py -3.13 -m pip install pandas jupyter
```

In [23]:
pip install pandas jupyter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from pathlib import Path
import re
import pandas as pd

RAW_CSV = Path('ofbiz_hr_tasks.csv')       # Preferred: live Jira collection
FALLBACK_CSV = Path('ofbiz_hr_tasks_cleaned.csv')  # Earlier cleaned scrape, if raw file was removed
REFERENCE_CSV = Path('dataset.csv')        # Existing project format
OUTPUT_CSV = Path('ofbiz_hr_tasks_preprocessed.csv')

EXPECTED_COLUMNS = [
    'Status', 'Priority', 'Created', 'Resolved', 'Labels', 'Description',
    'title', 'task_type', 'assignee_id', 'lead_time_days',
]

# Conservative HR terms: avoids generic i18n, licensing, and translation tasks.
HR_TERMS = (
    'leave', 'attendance', 'employee', 'employment', 'human resource', 'humanres',
    'recruit', 'applicant', 'interview', 'job requisition', 'job posting', 'payroll',
    'holiday', 'timesheet', 'time sheet', 'employee position', 'reporting structure',
    'benefit', 'personnel', 'performance review', 'training',
)


In [25]:
# Load all fields as text so identifiers and dates are not changed accidentally.
INPUT_CSV = RAW_CSV if RAW_CSV.exists() else FALLBACK_CSV
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Run the scraper first: {RAW_CSV}')

raw_df = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False, encoding='utf-8')
if list(raw_df.columns) != EXPECTED_COLUMNS:
    raise ValueError(f'Unexpected input columns: {list(raw_df.columns)}')

print(f'Input file: {INPUT_CSV} | rows: {len(raw_df)}')
display(raw_df.head(3))

Input file: ofbiz_hr_tasks.csv | rows: 64


,Status,Priority,Created,Resolved,Labels,Description,title,task_type,assignee_id,lead_time_days
0,Done,Major,2006-07-27,2007-04-21,,There is a need to be able to block viewing in...,Roles and Security for Display of data.,Improvement,,268
1,Done,Major,2007-05-21,2007-06-01,,"We have created the Services,Forms and Screens...","Create Services,Forms and Screens for HumanRes...",Improvement,ofbiz_hr_19a8583262f9d051,11
2,Done,Major,2007-06-01,2007-06-01,,"This patch will be help to load ""humanres"" and...",Default Component Load using index.jsp,Improvement,ofbiz_hr_7b7a27c7d781ce7c,0


## 2. Identify HR-topic records

The search is limited to the Jira `humanres` component, but historic records can still contain unrelated maintenance. This filter checks title, description, and labels for HR feature terms. Review the preview before exporting.

In [26]:
search_text = (
    raw_df['title'].fillna('') + ' ' + raw_df['Description'].fillna('') + ' ' + raw_df['Labels'].fillna('')
).str.casefold()
topic_pattern = '|'.join(re.escape(term) for term in HR_TERMS)
topic_mask = search_text.str.contains(topic_pattern, regex=True, na=False)

df = raw_df.loc[topic_mask].copy()
print(f'Kept HR-topic rows: {len(df)}')
print(f'Filtered unrelated rows: {len(raw_df) - len(df)}')
display(df[['title', 'Labels', 'task_type']].head(10))

Kept HR-topic rows: 64
Filtered unrelated rows: 0


,title,Labels,task_type
0,Roles and Security for Display of data.,,Improvement
1,"Create Services,Forms and Screens for HumanRes...",,Improvement
2,Default Component Load using index.jsp,,Improvement
3,New Styles For Human Resources application,,Improvement
4,Improve usability of Employment Screen,,Feature Development
5,Improve usability of Reporting Structure Screen,,Improvement
6,Minor improvement to Employee Position Form,,Sub-task
7,Add Leave header Item in HR module,,Sub-task
8,"Re-organize RateType, rateAmount and rateCurre...",,Improvement
9,Error in HumanRes,,Bug Fix


## 3. Clean, deduplicate, and validate

This standardizes line endings and labels, removes duplicate task records, validates ISO dates, and recalculates non-negative lead time. Jira priorities remain unchanged. `ofbiz_hr_...` assignee IDs remain separate from the numeric synthetic IDs already in `dataset.csv`.

In [27]:
def clean_text(value):
    return str(value or '').replace('\r\n', '\n').replace('\r', '\n').strip()

GREETING = re.compile(r'^(hello|hi|dear)\b', re.IGNORECASE)
SIGN_OFF = re.compile(r'^(thanks?(?:\s*(?:&|and)\s*regards)?|regards|best regards|kind regards|thank you)[\s,!.]*$', re.IGNORECASE)

def clean_description(value):
    kept = []
    for line in clean_text(value).replace('\u00a0', ' ').split('\n'):
        line = re.sub(r'\s+', ' ', line).strip()
        if not line or GREETING.match(line):
            continue
        if SIGN_OFF.match(line):
            break  # Ignore a name after an email sign-off.
        kept.append(line)
    return ' '.join(kept)

def clean_labels(value):
    seen, labels = set(), []
    for label in str(value or '').split(','):
        label = clean_text(label)
        if label and label not in seen:
            labels.append(label)
            seen.add(label)
    return ','.join(labels)

for column in ['Status', 'Priority', 'title', 'task_type']:
    df[column] = df[column].map(clean_text)
df['Description'] = df['Description'].map(clean_description)
df['Labels'] = df['Labels'].map(clean_labels)

# Parse both dates with the same calendar-date rule.
created_dt = pd.to_datetime(df['Created'], errors='coerce').dt.normalize()
resolved_dt = pd.to_datetime(df['Resolved'], errors='coerce').dt.normalize()
valid_dates = created_dt.notna() & resolved_dt.notna() & (resolved_dt >= created_dt)
invalid_rows = int((~valid_dates).sum())
df = df.loc[valid_dates].copy()
df['Created'] = created_dt.loc[valid_dates].dt.strftime('%Y-%m-%d')
df['Resolved'] = resolved_dt.loc[valid_dates].dt.strftime('%Y-%m-%d')
df['lead_time_days'] = (resolved_dt.loc[valid_dates] - created_dt.loc[valid_dates]).dt.days.astype(str)

before_dedup = len(df)
df = df.drop_duplicates(subset=['title', 'Description', 'Created', 'Resolved'], keep='first')
print(f'Removed invalid date rows: {invalid_rows}')
print(f'Removed duplicate rows: {before_dedup - len(df)}')
print(f'Clean rows ready for export: {len(df)}')

Removed invalid date rows: 0
Removed duplicate rows: 0
Clean rows ready for export: 64


In [28]:
# Check schema compatibility with the existing project dataset.
if REFERENCE_CSV.exists():
    reference_columns = list(pd.read_csv(REFERENCE_CSV, nrows=0, encoding='utf-8-sig').columns)
    assert reference_columns == EXPECTED_COLUMNS, 'dataset.csv has a different schema'

assert list(df[EXPECTED_COLUMNS].columns) == EXPECTED_COLUMNS
assert (pd.to_numeric(df['lead_time_days']) >= 0).all()
required = ['Status', 'Priority', 'Created', 'Resolved', 'Description', 'title', 'task_type', 'lead_time_days']
print('Blank required fields:', {column: int(df[column].eq('').sum()) for column in required})
display(df[EXPECTED_COLUMNS].head(5))

Blank required fields: {'Status': 0, 'Priority': 0, 'Created': 0, 'Resolved': 0, 'Description': 4, 'title': 0, 'task_type': 0, 'lead_time_days': 0}


,Status,Priority,Created,Resolved,Labels,Description,title,task_type,assignee_id,lead_time_days
0,Done,Major,2006-07-27,2007-04-21,,There is a need to be able to block viewing in...,Roles and Security for Display of data.,Improvement,,268
1,Done,Major,2007-05-21,2007-06-01,,"We have created the Services,Forms and Screens...","Create Services,Forms and Screens for HumanRes...",Improvement,ofbiz_hr_19a8583262f9d051,11
2,Done,Major,2007-06-01,2007-06-01,,"This patch will be help to load ""humanres"" and...",Default Component Load using index.jsp,Improvement,ofbiz_hr_7b7a27c7d781ce7c,0
3,Done,Minor,2008-01-24,2008-01-25,,Apply the new CSS styles to all the forms into...,New Styles For Human Resources application,Improvement,ofbiz_hr_294c5a14c27a8aa5,1
4,Done,Minor,2008-02-19,2008-06-11,,Following things can be done to improve usabil...,Improve usability of Employment Screen,Feature Development,ofbiz_hr_d384a81093f4bb66,113


## 4. Save only the validated data

This final cell writes UTF-8 CSV. It does not change `dataset.csv`; review the output first, then append it separately only if approved.

In [29]:
df[EXPECTED_COLUMNS].to_csv(OUTPUT_CSV, index=False, encoding='utf-8', lineterminator='\n')
print(f'Saved {len(df)} validated rows to: {OUTPUT_CSV.resolve()}')

Saved 64 validated rows to: D:\Y4\ML\Scraping\ofbiz_hr_tasks_preprocessed.csv
